In [1]:
import pandas as pd
import numpy as np
from numpy.random import Generator, PCG64
from joblib import Parallel, delayed
import altair as alt

rng = Generator(PCG64())

In [2]:
all_df = pd.read_csv("./dataframes/all_evaluated_responses_df.csv")
all_df = all_df.dropna(subset=['agent_response'])
all_df['item_id'] = all_df['question'] + all_df['image_file']

In [3]:
model_configs = [
    {'model': "Salesforce/blip2-flan-t5-xl", 'top_p': 'p06', 'temp': 't1', 'color': '#5ba3cf', 'accent_color': '#8cbedd'},
    {'model': "Salesforce/blip2-flan-t5-xxl", 'top_p': 'p1', 'temp': 't10', 'color': '#4c78a8', 'accent_color': '#81a0c2'},
    {'model': "llava-hf/llava-1.5-7b-hf", 'top_p': 'p04', 'temp': 't1', 'color': '#f9b574', 'accent_color': '#facb9d'},
    {'model': "llava-hf/llava-1.5-13b-hf", 'top_p': 'p1', 'temp': 't04', 'color': '#f58518', 'accent_color': '#f79d46'},
    {'model': "liuhaotian/llava-v1.6-34b", 'top_p': 'p1', 'temp': 't04', 'color': '#BF9000', 'accent_color': '#F1C232'},
    {'model': "google/pix2struct-chartqa-base", 'top_p': 'p08', 'temp': 't1', 'color': '#ba94c4', 'accent_color': '#c7b8d9'},
    {'model': "google/matcha-chartqa", 'top_p': 'p04', 'temp': 't1', 'color': '#8b6db2', 'accent_color': '#a28ac1'},
    {'model': "GPT-4V", 'top_p': 'p1', 'temp': 't02', 'color': '#c93739', 'accent_color': '#C6765E'},
    {'model': "Human", 'top_p': 'pna', 'temp': 'tna', 'color': '#2e693b', 'accent_color': '#73D287'},
]

# Create Item DF

### Chart Type Labels

In [4]:
chart_type_maps = {
    'Line Chart': "line", 
    'Pie Chart': "pie", 
    'Bar Chart': "bar", 
    'Dot Plot': "dot-plot", 
    'Scatterplot': 'scatter',
    'Bubble Chart': 'bubble', 
    'Stacked Bar Chart': "stacked-bar", 
    'Stacked Area Chart': "stacked-area",
    'Treemap': 'treemap', 
    'Histogram': 'histogram', 
    'Choropleth Map': 'map', 
    'Area Chart': 'area',
    '100 % Stacked Bar Chart': '100-stacked-bar', 
    'Stacked area chart': 'stacked-area', 
    'Bar chart': 'bar',
    'Stacked bar chart': 'stacked-bar', 
    'Area chart': 'area', 
    'Choropleth map': 'map', 
    'Line chart': 'line',
    'Pie chart': 'pie', 
    '100% stacked bar chart': '100-stacked-bar',
    'h_bar_PEW': 'bar',
    'v_bar_two_var': 'bar', 
    'h_bar_two_var': 'bar', 
    'pie_PEW': 'pie', 
    'v_bar_multi_var': 'bar',
    'line_two_var': 'line', 
    'line_multi_var': 'line', 
    'pie_Statista': 'pie',
    'h_bar_multi_var': 'bar', 
    'line_OWID': 'line',
    'barchart': 'bar',
}

### Task Type Labels

In [5]:
vlat_format_map = {
    'find_extremum': 'value identification', 
    'determine_Range': 'value identification', 
    'retrieve_value': 'value identification',
    'make_comparisons': 'arithmetic computation',
    'find_correlations_trends': 'statistical inference', 
    'characterize_distribution': 'statistical inference',
    'find_anomolies': 'statistical inference', 
    'find_clusters': 'statistical inference', 
}

calvi_format_map = {
    'Make Comparisons': 'arithmetic computation',
    'Find Correlations/Trends': 'statistical inference', 
    'Find Extremum': 'value identification',
    'Retrieve Value': 'value identification', 
    'Make Predictions': 'statistical inference', 
    'Aggregate Values': 'statistical inference'
}

ggr_question_map = {
    # from level 1
    "Approximately what percentage of people had Adeolitis in the year 2000?": "value identification",
    "What percentage of patients recovered after chemotherapy?": "value identification",
    "Of 100 patients with disease X, how many are women?": "value identification",
    "Of all the people who die from cancer, approximately what percentage dies from lung cancer?": "value identification",

    # from level 2
    "Approximately what percentage of people who die from cancer die from colon cancer, breast cancer, and prostate cancer taken together?": "arithmetic computation",
    "How many more men than women are there among 100 patients with disease X?": "arithmetic computation",
    "What is the difference between the percentage of patients who recovered after a surgery and the percentage of patients who recovered after radiation therapy?": "arithmetic computation",
    "When was the increase in the percentage of people with Adeolitis higher?": "arithmetic computation",

    # from level 3
    "What percentage of patients recovered after chemotherapy?": "value identification",
    "According to your best guess, what will the percentage of people with Adeolitis be in the year 2010?": "statistical inference",
    "Between 1980 and 1990, which disease had a higher increase in the percentage of people affected?": "statistical inference",
    "Compared to the placebo, which treatment leads to a larger decrease in the percentage of patients who die?": "statistical inference",
    "What is the percentage of cancer patients who die after chemotherapy?": "value identification",
    "Which of the treatments contributes to a larger decrease in the percentage of sick patients?": "statistical inference"
}

holf_map = {
    'retrieve_value': 'value identification', 
    'make_comparisons': 'arithmetic computation',
    'determine_range': 'arithmetic computation',
}

holf2_map = {
    'retrieve_value': 'value identification', 
    'make_comparisons': 'arithmetic computation',
    'predict_trend': 'statistical inference'
}

chartqa_map = {
    'retrive value': 'value identification'
}


task_category_maps = vlat_format_map | calvi_format_map | holf_map | holf2_map | chartqa_map

### Create item property dataframe

In [6]:
def create_item_df():
    tests = [
        'ggr', 
        'vlat', 
        'calvi', 
        'holf', 
        'holf2',
        'chartqa-test-continuous-human'
    ]
    
    item_df = []
    for test in tests:
        idf = pd.read_csv(f'https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/{test}/questions.csv')
        idf['test_type'] = test

        if test == 'ggr':
            idf['task_category'] = idf['question'].replace(ggr_question_map)
        else:
            idf['task_category'] = idf['task_category'].replace(task_category_maps)

        item_df.append(idf)

    item_df = pd.concat(item_df)
    item_df['question_image'] = item_df['question'] + " + " + item_df['image_file']
    return item_df

item_df = create_item_df()
item_df['item_id'] = item_df['question'] + item_df['image_file']
item_df['chart_type'] = item_df['chart_type'].replace(chart_type_maps)

print(item_df['chart_type'].unique())
print(item_df['task_category'].unique())

['bar' 'line' 'pie' 'dot-plot' 'stacked-bar' '100-stacked-bar' 'histogram'
 'scatter' 'area' 'stacked-area' 'bubble' 'map' 'treemap']
['value identification' 'statistical inference' 'arithmetic computation']


## Feature Pairwise Figure

In [43]:
# group_average_df = all_df.groupby(['agent_type', 'test_type', 'item_id'])['is_correct'].mean().reset_index()
group_average_df = all_df[all_df['agent_type'].isin(['GPT-4V', 'Human'])]
group_average_df = pd.merge(group_average_df, item_df[['item_id', 'task_category', 'chart_type']])
group_average_df['test_type'] = group_average_df['test_type'].replace({
    'ggr': 'mc', 
    'vlat': 'mc', 
    'calvi': 'mc',
    'holf': 'num', 
    'holf2': 'num', 
    'chartqa-test-continuous-human': 'num'
})

task_group_average_df = group_average_df.groupby(['agent_type', 'test_type', 'task_category'])['is_correct'].mean().reset_index()
graph_group_average_df = group_average_df.groupby(['agent_type', 'test_type', 'chart_type'])['is_correct'].mean().reset_index()

In [45]:
# task_group_average_df

### Pairwise Slope Plots

In [9]:
domain = ['GPT-4V', 'Human']
color_domain = [model_configs[-2]['color'], model_configs[-1]['color']]

tests = ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']

charts =[]
graph_group_average_simplified = graph_group_average_df.copy().replace({
    'ggr': 'mc', 
    'vlat': 'mc', 
    'calvi': 'mc',
    'holf': 'num', 
    'holf2': 'num', 
    'chartqa-test-continuous-human': 'num'
})

graph_group_average_simplified = graph_group_average_simplified.groupby(['test_type', 'agent_type', 'chart_type'])['is_correct'].mean().reset_index()

for test in ['mc', 'num']:
    df = graph_group_average_simplified[graph_group_average_simplified['test_type'] == test]
    
    line_chart = alt.Chart(df).mark_line(opacity=0.8).encode(
        x=alt.X('agent_type:N', title=None),
        y=alt.Y('is_correct', scale=alt.Scale(domain=[0, 1]), title=None),
        detail='chart_type',
        # color=alt.value('black'),
        color=alt.Color('chart_type', scale=alt.Scale(scheme='category20'))
    ).properties(width=150, title=test)

    point_plot = alt.Chart(df).mark_circle(opacity=0.6).encode(
        x='agent_type:N',
        y='is_correct',
        # color=alt.Color('agent_type',legend=None, scale=alt.Scale(domain=domain, range=color_domain)),
        # color=alt.Color('test_type',legend=None, scale=alt.Scale(domain=test_domain, range=color_domain)),
        size=alt.value(40)
    )

    chart = (line_chart + point_plot).resolve_scale(color='shared').properties(width=100, height=200)

    charts.append(chart)

alt.hconcat(*charts)

alt.HConcatChart(...)

In [10]:
domain = ['GPT-4V', 'Human']
color_domain = [model_configs[-2]['color'], model_configs[-1]['color']]

tests = ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']

charts =[]
task_group_average_simplified = task_group_average_df.copy().replace({
    'ggr': 'mc', 
    'vlat': 'mc', 
    'calvi': 'mc',
    'holf': 'num', 
    'holf2': 'num', 
    'chartqa-test-continuous-human': 'num'
})

task_group_average_simplified = task_group_average_simplified.groupby(['test_type', 'agent_type', 'task_category'])['is_correct'].mean().reset_index()

for test in ['mc', 'num']:
    df = task_group_average_simplified[task_group_average_simplified['test_type'] == test]
    
    line_chart = alt.Chart(df).mark_line(opacity=0.8).encode(
        x=alt.X('agent_type:N', title=None),
        y=alt.Y('is_correct', scale=alt.Scale(domain=[0, 1]), title=None),
        detail='task_category',
        # color=alt.value('black'),
        color=alt.Color('task_category')
    ).properties(width=150, title=test)

    point_plot = alt.Chart(df).mark_circle(opacity=0.6).encode(
        x='agent_type:N',
        y='is_correct',
        # color=alt.Color('agent_type',legend=None, scale=alt.Scale(domain=domain, range=color_domain)),
        # color=alt.Color('test_type',legend=None, scale=alt.Scale(domain=test_domain, range=color_domain)),
        size=alt.value(40)
    )

    chart = (line_chart + point_plot).resolve_scale(color='shared').properties(width=100, height=200)

    charts.append(chart)

alt.hconcat(*charts)

alt.HConcatChart(...)

### Graph Group

In [37]:
# graph_group_average_df[
#     graph_group_average_df['agent_type'] == 'Human'
# ]

In [46]:
icon_graph_maps = pd.DataFrame([
    {'chart_type': '100-stacked-bar', 'icon': 'norm_stacked_bar.svg'}, 
    {'chart_type': 'area', 'icon': 'area.svg'}, 
    {'chart_type': 'bar', 'icon': 'bar.svg'}, 
    {'chart_type': 'bubble', 'icon': 'bubble.svg'}, 
    {'chart_type': 'dot-plot', 'icon': 'dot_plot.svg'},
    {'chart_type': 'histogram', 'icon': 'histogram.svg'}, 
    {'chart_type': 'line', 'icon': 'line.svg'}, 
    {'chart_type': 'map', 'icon': 'map.svg'}, 
    {'chart_type': 'pie', 'icon': 'pie.svg'}, 
    {'chart_type': 'scatter', 'icon': 'scatter.svg'}, 
    {'chart_type': 'stacked-area', 'icon': 'stacked_area.svg'},
    {'chart_type': 'stacked-bar', 'icon': 'stacked_bar.svg'}, 
    {'chart_type': 'treemap', 'icon': 'treemap.svg'}
])

icon_graph_maps['icon'] =  "https://raw.githubusercontent.com/vermaarn/img_store/main/" + icon_graph_maps['icon']
icon_graph_maps['icon'] = icon_graph_maps['icon'].apply(lambda l : l.replace('.svg', '.png'))

In [39]:
# graph_group_average_simplified = graph_group_average_df.replace({
#     'ggr': 'mc', 
#     'vlat': 'mc', 
#     'calvi': 'mc',
#     'holf': 'num', 
#     'holf2': 'num', 
#     'chartqa-test-continuous-human': 'num'
# })

In [52]:
def create_graph_dataframe(graph_group_average_df):
    mc_graphs = graph_group_average_df[
        graph_group_average_df['test_type'] == 'mc'
    ].pivot(index='chart_type',columns=['agent_type'], values='is_correct').reset_index()
    mc_graphs['test_type'] = 'mc'
    
    num_graphs = graph_group_average_df[
        graph_group_average_df['test_type'] == 'num'
    ].pivot(index='chart_type',columns=['agent_type'], values='is_correct').reset_index()
    num_graphs['test_type'] = 'num'
    
    graph_group = pd.concat([mc_graphs, num_graphs])
    graph_group = pd.merge(graph_group, icon_graph_maps)
    return graph_group

In [59]:
graph_group = create_graph_dataframe(graph_group_average_df)

In [75]:


gpt_human_graph_performance_img_icon = alt.Chart(graph_group).mark_image(width=15, height=15).encode(
    x=alt.X('Human', scale=alt.Scale(domain=[0,1])),
    y=alt.Y('GPT-4V', scale=alt.Scale(domain=[0,1])),
    url=alt.Url('icon'),
    color=alt.value('black'),
    opacity=alt.value(0.5)
).properties(height=300, width=300)

gpt_human_graph_performance_circle = alt.Chart(graph_group).mark_point(size=100, filled=True).encode(
    x=alt.X('Human', scale=alt.Scale(domain=[0,1])),
    y=alt.Y('GPT-4V', scale=alt.Scale(domain=[0,1])),
    # opacity=alt.Opacity('test_type', scale=alt.Scale(domain=['mc', 'num'], range=[1, 0.4])),
    shape=alt.Shape('test_type', scale=alt.Scale(domain=['mc', 'num'], range=['circle', 'square'])),
    color=alt.value('black')
).properties(height=300, width=300)

line_layer = alt.Chart(
    pd.DataFrame({'x': [0, 1], 'y': [0, 1]})
).mark_line(strokeWidth=2, strokeDash=[5, 5]).encode(
    x='x:Q',
    y='y:Q',
    color=alt.value('black')
)

graph_performance = gpt_human_graph_performance_circle
graph_performance += gpt_human_graph_performance_img_icon
graph_performance += line_layer

graph_performance

alt.LayerChart(...)

### Task Group

In [229]:
domain = ['GPT-4V', 'Human']
color_domain = [model_configs[-2]['color'], model_configs[-1]['color']]

tests = ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']

charts =[]
task_group_average_simplified = task_group_average_df.copy().replace({
    'ggr': 'mc', 
    'vlat': 'mc', 
    'calvi': 'mc',
    'holf': 'num', 
    'holf2': 'num', 
    'chartqa-test-continuous-human': 'num'
})

task_group_average_simplified = task_group_average_simplified.groupby(['test_type', 'agent_type', 'task_category'])['is_correct'].mean().reset_index()

icon_task_maps = pd.DataFrame([
    {'task_category': 'arithmetic computation', 'icon': 'arithmetic_computation.svg'}, 
    {'task_category': 'statistical inference', 'icon': 'statistical_inference1.svg'}, 
    {'task_category': 'value identification', 'icon': 'value_identification.svg'}, 
])
icon_task_maps['icon'] =  "https://raw.githubusercontent.com/vermaarn/img_store/main/" + icon_task_maps['icon']
icon_task_maps['icon'] = icon_task_maps['icon'].apply(lambda l : l.replace('.svg', '.png'))

In [230]:
mc_tasks = task_group_average_simplified[
    task_group_average_simplified['test_type'] == 'mc'
].pivot(index='task_category',columns=['agent_type'], values='is_correct').reset_index()
mc_tasks['test_type'] = 'mc'

num_tasks = task_group_average_simplified[
    task_group_average_simplified['test_type'] == 'num'
].pivot(index='task_category',columns=['agent_type'], values='is_correct').reset_index()
num_tasks['test_type'] = 'num'

task_group = pd.concat([mc_tasks, num_tasks])
task_group = pd.merge(task_group, icon_task_maps)

In [231]:
task_pairwise_figure = alt.Chart(task_group).mark_image(width=15, height=15).encode(
    x=alt.X('Human', scale=alt.Scale(domain=[0,1])),
    y=alt.Y('GPT-4V', scale=alt.Scale(domain=[0,1])),
    url=alt.Url('icon'),
    opacity=alt.Opacity('test_type', scale=alt.Scale(domain=['mc', 'num'], range=[1, 0.6])),
    color=alt.value('black')
).properties(height=300, width=300)

line_layer = alt.Chart(
    pd.DataFrame({'x': [0, 1], 'y': [0, 1]})
).mark_line(strokeWidth=2, strokeDash=[5, 5]).encode(
    x='x:Q',
    y='y:Q',
    color=alt.value('black')
)
task_pairwise_figure += line_layer
task_pairwise_figure

alt.LayerChart(...)

In [234]:
feature_performance = alt.hconcat(graph_pairwise_figure, task_pairwise_figure)
feature_performance.save("./figures/feature_performance.pdf")

feature_performance

alt.HConcatChart(...)

In [233]:
# icon_task_maps.iloc[0]['icon']

In [81]:
# graph_group_average_simplified

,agent_type,test_type,chart_type,is_correct
0,GPT-4V,mc,100-stacked-bar,0.625000
1,GPT-4V,mc,area,0.529412
2,GPT-4V,mc,bar,0.553846
3,GPT-4V,mc,line,0.416667
4,GPT-4V,mc,map,0.386667
...,...,...,...,...
63,Human,mc,pie,0.874890
64,Human,mc,scatter,0.770296
65,Human,mc,stacked-area,0.521586
66,Human,mc,stacked-bar,0.670661
